# BPNL(Buy Now Pay Later) Risk Scoring & Analytics (India)

**| Applying Machine Learning Models**

I. Loading Modules.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

II. Loading the CSV File.

In [2]:
df = pd.read_csv('/content/bpnl.csv').sort_values('S_no')
df

,S_no,User_id,Lending_company,Age,City,State,Monthly_income,Loan_amount,Reason,Credit_limit,Tenure_months,Year,Transactions_per_month,Missed_Payments,Repayment_Method,Credit_utilization_ratio,Default_risk
10733,1,233044,PaySense,25,Chennai,Tamil Nadu,36764,305304.631876,Vehicle Loan,24077,5,2025,7,2,Debit Card,0.461735,6
64628,2,894361,ZestMoney,25,Kochi,Kerala,13930,46380.811951,Vehicle Loan,29178,14,2022,5,4,Credit Card,0.625266,3
40323,3,594950,ZestMoney,22,Ahmedabad,Gujarat,75487,249823.904759,Home P&C Loan,30431,18,2024,5,0,Credit Card,0.452116,8
55054,4,775841,Simpl,46,Kochi,Kerala,33552,85540.457923,Electronics,61093,5,2024,24,4,Net Banking,0.471355,7
67029,5,925283,LazyPay,27,Lucknow,Uttar Pradesh,56777,123562.123184,Travel & Vacation,91339,19,2024,24,1,UPI/Wallet,0.490594,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6502,73206,180378,LazyPay,58,Delhi,Delhi,25356,212065.704057,Home P&C Loan,75273,1,2023,24,2,UPI/Wallet,0.144292,3
63409,73207,879268,Simpl,50,Delhi,Delhi,87627,208170.632703,Home P&C Loan,71387,20,2023,30,2,Net Banking,0.548311,6
69350,73208,953871,ZestMoney,19,Coimbatore,Tamil Nadu,29357,113130.847521,Electronics,23499,16,2025,9,1,Debit Card,0.500213,9
30752,73209,478258,PaySense,22,Pune,Maharashtra,37983,136767.529468,Education Loan,40127,20,2022,7,1,UPI/Wallet,0.856134,8


III. Adding Age Category Column.

In [3]:
df['Age_Category'] = df['Age'].apply(
    lambda x: 'Gen Z' if 18 <= x <= 25
    else 'Millennial' if 26 <= x <= 40
    else 'Boomer' if 41 <= x <= 60
    else 'Old Age')
df

,S_no,User_id,Lending_company,Age,City,State,Monthly_income,Loan_amount,Reason,Credit_limit,Tenure_months,Year,Transactions_per_month,Missed_Payments,Repayment_Method,Credit_utilization_ratio,Default_risk,Age_Category
10733,1,233044,PaySense,25,Chennai,Tamil Nadu,36764,305304.631876,Vehicle Loan,24077,5,2025,7,2,Debit Card,0.461735,6,Gen Z
64628,2,894361,ZestMoney,25,Kochi,Kerala,13930,46380.811951,Vehicle Loan,29178,14,2022,5,4,Credit Card,0.625266,3,Gen Z
40323,3,594950,ZestMoney,22,Ahmedabad,Gujarat,75487,249823.904759,Home P&C Loan,30431,18,2024,5,0,Credit Card,0.452116,8,Gen Z
55054,4,775841,Simpl,46,Kochi,Kerala,33552,85540.457923,Electronics,61093,5,2024,24,4,Net Banking,0.471355,7,Boomer
67029,5,925283,LazyPay,27,Lucknow,Uttar Pradesh,56777,123562.123184,Travel & Vacation,91339,19,2024,24,1,UPI/Wallet,0.490594,4,Millennial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6502,73206,180378,LazyPay,58,Delhi,Delhi,25356,212065.704057,Home P&C Loan,75273,1,2023,24,2,UPI/Wallet,0.144292,3,Boomer
63409,73207,879268,Simpl,50,Delhi,Delhi,87627,208170.632703,Home P&C Loan,71387,20,2023,30,2,Net Banking,0.548311,6,Boomer
69350,73208,953871,ZestMoney,19,Coimbatore,Tamil Nadu,29357,113130.847521,Electronics,23499,16,2025,9,1,Debit Card,0.500213,9,Gen Z
30752,73209,478258,PaySense,22,Pune,Maharashtra,37983,136767.529468,Education Loan,40127,20,2022,7,1,UPI/Wallet,0.856134,8,Gen Z


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 73210 entries, 10733 to 884
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   S_no                      73210 non-null  int64  
 1   User_id                   73210 non-null  int64  
 2   Lending_company           73210 non-null  object 
 3   Age                       73210 non-null  int64  
 4   City                      73210 non-null  object 
 5   State                     73210 non-null  object 
 6   Monthly_income            73210 non-null  int64  
 7   Loan_amount               73210 non-null  float64
 8   Reason                    73210 non-null  object 
 9   Credit_limit              73210 non-null  int64  
 10  Tenure_months             73210 non-null  int64  
 11  Year                      73210 non-null  int64  
 12  Transactions_per_month    73210 non-null  int64  
 13  Missed_Payments           73210 non-null  int64  
 14  Repayment

In [5]:
df_numerical = ['Age','Monthly_income','Loan_amount','Credit_limit',
                     'Tenure_months','Transactions_per_month','Missed_Payments',
                     'Credit_utilization_ratio','Default_risk']
df_numerical

['Age',
 'Monthly_income',
 'Loan_amount',
 'Credit_limit',
 'Tenure_months',
 'Transactions_per_month',
 'Missed_Payments',
 'Credit_utilization_ratio',
 'Default_risk']

IV. Linear Regression.

In [6]:
#Importing nessesary models & applying Label Encoding on Categorial columns.

import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df=df.drop(['S_no','City','State'],axis=1, errors='ignore')

le = LabelEncoder()
df['Age_Category'] = le.fit_transform(df['Age_Category'])
df['Reason'] = le.fit_transform(df['Reason'])
df['Repayment_Method'] = le.fit_transform(df['Repayment_Method'])
df['Lending_company'] = le.fit_transform(df['Lending_company'])

In [7]:
df

,User_id,Lending_company,Age,Monthly_income,Loan_amount,Reason,Credit_limit,Tenure_months,Year,Transactions_per_month,Missed_Payments,Repayment_Method,Credit_utilization_ratio,Default_risk,Age_Category
10733,233044,6,25,36764,305304.631876,5,24077,5,2025,7,2,1,0.461735,6,1
64628,894361,9,25,13930,46380.811951,5,29178,14,2022,5,4,0,0.625266,3,1
40323,594950,9,22,75487,249823.904759,2,30431,18,2024,5,0,0,0.452116,8,1
55054,775841,7,46,33552,85540.457923,1,61093,5,2024,24,4,2,0.471355,7,0
67029,925283,4,27,56777,123562.123184,4,91339,19,2024,24,1,3,0.490594,4,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6502,180378,4,58,25356,212065.704057,2,75273,1,2023,24,2,3,0.144292,3,0
63409,879268,7,50,87627,208170.632703,2,71387,20,2023,30,2,2,0.548311,6,0
69350,953871,9,19,29357,113130.847521,1,23499,16,2025,9,1,1,0.500213,9,1
30752,478258,6,22,37983,136767.529468,0,40127,20,2022,7,1,3,0.856134,8,1


1. Linear Regression on Loan Amount Column

In [8]:
x = df.drop('Loan_amount',axis=1)
y = df['Loan_amount']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=420)

model=LinearRegression()
model.fit(x_train, y_train)

y_pred= model.predict(x_test)
y_pred

array([218702.22370546,  50128.46039478, 153095.0627545 , ...,
       279155.80446052,   7945.27870671, 264392.09806927])

In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, y_pred))

Mean Squared Error: 6860405443.263509
Mean Absolute Error: 70995.73740992486
R2 Score: 0.3906789038677926
Root Mean Squared Error: 82827.56451365395


2. Linear Regression on Default Risk Column.

In [10]:
x = df.drop('Default_risk',axis=1)
y = df['Default_risk']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=420)

model=LinearRegression()
model.fit(x_train, y_train)

y_pred= model.predict(x_test)
y_pred

array([4.97821208, 4.93218846, 5.02023058, ..., 4.99894555, 4.95508746,
       4.93830077])

In [11]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, y_pred))

Mean Squared Error: 6.705240580843137
Mean Absolute Error: 2.2329374128654167
R2 Score: -0.0002790169553852362
Root Mean Squared Error: 2.589447929741615


3. Linear Regression on Credit Utilization Ratio Column.

In [12]:
x = df.drop('Credit_utilization_ratio',axis=1)
y = df['Credit_utilization_ratio']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=420)

model=LinearRegression()
model.fit(x_train, y_train)

y_pred= model.predict(x_test)
y_pred

array([0.54207463, 0.54386687, 0.52257768, ..., 0.54526143, 0.5212974 ,
       0.55843461])

In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, y_pred))

Mean Squared Error: 0.0621716080731689
Mean Absolute Error: 0.2146567063097139
R2 Score: 0.0024027357625061585
Root Mean Squared Error: 0.24934235114229772


4. Linear Regression on Age Category Column.

In [14]:
x = df.drop('Age_Category',axis=1)
y = df['Age_Category']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=420)

model=LinearRegression()
model.fit(x_train, y_train)

y_pred= model.predict(x_test)
y_pred

array([1.15693315, 1.55857609, 0.97434864, ..., 1.48979322, 1.92596239,
       1.41766221])

In [15]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, y_pred))

Mean Squared Error: 0.6760948286268266
Mean Absolute Error: 0.6804758609637439
R2 Score: 0.09898937210115022
Root Mean Squared Error: 0.8222498577846192


5. Linear Regression on Lending Company Column.

In [16]:
x = df.drop('Lending_company',axis=1)
y = df['Lending_company']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=420)

model=LinearRegression()
model.fit(x_train, y_train)

y_pred= model.predict(x_test)
y_pred

array([6.73339229, 7.81806184, 8.34817428, ..., 6.12854109, 8.07191075,
       6.28112616])

In [17]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, y_pred))

Mean Squared Error: 3.519055172640112
Mean Absolute Error: 1.540252025227439
R2 Score: 0.16532672715334562
Root Mean Squared Error: 1.8759144896929902
